# 04. 최종 모델

용어는 [GLOSSARY.md](../GLOSSARY.md)를 따른다.

03이 남긴 실행 결과로 **최종 모델**을 고르고 마무리한다.

1. 최종 모델 선택: 2단계 학습 시드 중 val macro F1 최고 (비교 결과와 관계없이 최종 모델은 2단계 학습)
2. 범주별 threshold 튜닝 (val에서만) → test에서 0.5와 비교, 배타 규칙 전후·판정 없음 보고
3. 리뷰 예측 함수와 NSMC 정성 데모
4. NSMC 극성 일관성 점검

**실행 전에**: 03을 끝까지 실행해 Drive `MyDrive/miniproject/kote-runs/`에 결과가 있어야 한다. 1~2절은 CPU로 돌고, 3~4절은 GPU가 있으면 빠르다.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    !pip -q install transformers scikit-learn

In [ ]:
import os

# SMOKE=1 이면 03의 SMOKE 결과로 코드가 끝까지 도는지만 확인한다.
SMOKE = os.environ.get("SMOKE") == "1"

CATEGORIES = ["기쁨", "슬픔", "분노", "공포", "놀람", "혐오", "지루함", "중립"]
NEUTRAL = CATEGORIES.index("중립")
SEEDS = [42, 43] if SMOKE else [42, 43, 44]   # 03과 같아야 한다
METHODS = ["direct", "two_stage"]
MAX_LEN = 128

if IN_COLAB and not SMOKE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = "/content/drive/MyDrive/miniproject"
else:
    DRIVE_DIR = "outputs/smoke" if SMOKE else "outputs"

RUNS_DIR = f"{DRIVE_DIR}/kote-runs"
RESULTS_DIR = "outputs/smoke/results" if SMOKE else "outputs/results"  # git에 커밋할 요약 결과
os.makedirs(RESULTS_DIR, exist_ok=True)

def save_result(df, name, **kw):
    df.to_csv(f"{RESULTS_DIR}/{name}", **kw)
    df.to_csv(f"{RUNS_DIR}/{name}", **kw)  # Colab 로컬 파일은 세션과 함께 사라지므로 Drive에도

## 1. 최종 모델 선택

03 결과 폴더의 실행마다 확률과 그 행의 댓글 id가 있다. id로 KOTE 정답을 찾아 맞춘 뒤, 03과 같은 평가 정의로 다시 계산해 03의 기록과 같은지 확인한다.

In [ ]:
import json
import numpy as np
import pandas as pd

KOTE_URL = "https://raw.githubusercontent.com/tlgus2882-ctrl/nlp_miniproject/main/outputs/kote-8cat/{}.csv"

def load_kote(split):
    local = f"outputs/kote-8cat/{split}.csv"
    return pd.read_csv(local if os.path.exists(local) else KOTE_URL.format(split)).set_index("id")

kote = {s: load_kote(s) for s in ["val", "test"]}

NEEDED = ["metrics.json", "val_probs.npy", "test_probs.npy", "val_ids.npy", "test_ids.npy"]
missing = [f"{m}-seed{s}/{f}" for m in METHODS for s in SEEDS for f in NEEDED + (["model"] if m == "two_stage" else [])
           if not os.path.exists(f"{RUNS_DIR}/{m}-seed{s}/{f}")]
if missing:
    # 03은 metrics.json이 있으면 그 실행을 건너뛰므로, 해당 실행 폴더를 지워야 다시 학습한다
    raise FileNotFoundError(f"03 결과가 없습니다: {missing} → Drive에서 그 실행 폴더를 지우고 03을 다시 실행하세요")

runs = {}
for m in METHODS:
    for s in SEEDS:
        d = f"{RUNS_DIR}/{m}-seed{s}"
        runs[(m, s)] = {
            "metrics": json.load(open(f"{d}/metrics.json")),
            **{f"{sp}_{k}": np.load(f"{d}/{sp}_{k}.npy") for sp in ["val", "test"] for k in ["probs", "ids"]},
        }

# 모든 실행이 같은 댓글을 같은 순서로 평가했는지 확인하고, 그 순서로 정답을 꺼낸다
first = next(iter(runs.values()))
for r in runs.values():
    assert all(np.array_equal(r[f"{sp}_ids"], first[f"{sp}_ids"]) for sp in ["val", "test"])
labels = {sp: kote[sp].loc[first[f"{sp}_ids"], CATEGORIES].values.astype(int) for sp in ["val", "test"]}
print({sp: v.shape for sp, v in labels.items()})

In [ ]:
from sklearn.metrics import f1_score

def raw_predict(probs, threshold=0.5):
    return (probs > np.asarray(threshold)).astype(int)   # threshold: 0.5 하나 또는 범주별 8개

def apply_exclusive(preds):
    preds = preds.copy()
    other_on = np.delete(preds, NEUTRAL, axis=1).any(axis=1)
    preds[other_on, NEUTRAL] = 0
    return preds

def predict_labels(probs, threshold=0.5):
    return apply_exclusive(raw_predict(probs, threshold))

def score(labels, probs, threshold=0.5):
    preds = predict_labels(probs, threshold)
    per_cat = f1_score(labels, preds, average=None, zero_division=0)
    return {
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
        "no_prediction": (preds.sum(axis=1) == 0).mean(),
        **{f"f1_{c}": f for c, f in zip(CATEGORIES, per_cat)},
    }

# 회귀 확인: 0.5로 다시 계산한 지표가 03이 기록한 값과 같다 (평가 정의가 03과 같다)
for (m, s), r in runs.items():
    for sp in ["val", "test"]:
        assert np.isclose(score(labels[sp], r[f"{sp}_probs"])["macro_f1"], r["metrics"][f"{sp}_macro_f1"], atol=1e-6), (m, s, sp)
print("03 지표와 일치")

In [ ]:
METHOD_NAMES = {"direct": "직접 학습", "two_stage": "2단계 학습"}

table = pd.DataFrame([
    {"방식": METHOD_NAMES[m], "시드": s,
     "val macro F1": r["metrics"]["val_macro_f1"], "test macro F1": r["metrics"]["test_macro_f1"]}
    for (m, s), r in runs.items()
])

# 최종 모델 = 2단계 학습 중 val macro F1 최고 (동점이면 작은 시드). test는 고르는 데 쓰지 않는다
two = table[table["방식"] == "2단계 학습"].sort_values(["val macro F1", "시드"], ascending=[False, True])
FINAL_SEED = int(two.iloc[0]["시드"])

FINAL_DIR = f"{RUNS_DIR}/two_stage-seed{FINAL_SEED}/model"
final = runs[("two_stage", FINAL_SEED)]
table["최종 모델"] = np.where((table["방식"] == "2단계 학습") & (table["시드"] == FINAL_SEED), "◀", "")
print(f"최종 모델: 2단계 학습 seed {FINAL_SEED}")
table.round(4)

In [ ]:
summary = []

def violations(preds):
    return int((preds[:, NEUTRAL] & np.delete(preds, NEUTRAL, axis=1).any(axis=1)).sum())  # 중립과 다른 범주가 함께 켜진 행

def add_summary(name, threshold, run=None):
    run = run or final   # 07(pos_weight)은 다른 실행의 확률로 행을 추가한다
    row = {"설정": name}
    for sp in ["val", "test"]:
        row.update({f"{sp}_{k}": v for k, v in score(labels[sp], run[f"{sp}_probs"], threshold).items()})
        row[f"{sp}_rule_violations"] = violations(raw_predict(run[f"{sp}_probs"], threshold))  # 배타 규칙 적용 전
    summary.append(row)
    save_result(pd.DataFrame(summary), "final_summary.csv", index=False)

add_summary("기본(0.5)", 0.5)
pd.DataFrame(summary)[["설정", "val_macro_f1", "test_macro_f1", "test_micro_f1", "test_no_prediction"]].round(4)

## 2. 범주별 threshold 튜닝

- 최종 모델의 **val** 확률에서 범주마다 F1이 가장 높은 threshold를 고른다 (0.05~0.95, 0.05 간격). 범주별 F1은 배타 규칙 적용 전 예측으로 본다.
- test에는 고른 threshold를 한 번만 적용한다.
- 배타 규칙은 튜닝 뒤에 적용한다.

In [ ]:
GRID = np.round(np.arange(0.05, 0.951, 0.05), 2)

def tune_thresholds(val_labels, val_probs):
    # val만 받는다: test는 튜닝 경로에 들어오지 않는다
    return np.array([
        GRID[np.argmax([f1_score(val_labels[:, j], val_probs[:, j] > t, zero_division=0) for t in GRID])]
        for j in range(len(CATEGORIES))
    ])

THRESHOLDS = tune_thresholds(labels["val"], final["val_probs"])

add_summary("threshold 튜닝", THRESHOLDS)
save_result(pd.DataFrame({"감정 범주": CATEGORIES, "threshold": THRESHOLDS}), "final_thresholds.csv", index=False)
json.dump(dict(zip(CATEGORIES, THRESHOLDS.tolist())), open(f"{FINAL_DIR}/thresholds.json", "w"), ensure_ascii=False)
pd.DataFrame({"threshold": THRESHOLDS}, index=CATEGORIES).T

In [ ]:
cols = ["설정", "val_macro_f1", "test_macro_f1", "test_micro_f1", "test_no_prediction"] + [f"test_f1_{c}" for c in CATEGORIES]
pd.DataFrame(summary)[cols].round(4)

In [ ]:
# 배타 규칙 전후: 규칙 전 예측에서 중립과 다른 범주가 함께 켜진 test 행, 규칙 전후 중립 F1
rows = []
for name, th in [("기본(0.5)", 0.5), ("threshold 튜닝", THRESHOLDS)]:
    raw = raw_predict(final["test_probs"], th)
    after = apply_exclusive(raw)
    assert violations(after) == 0
    rows.append({
        "설정": name,
        "규칙 전 위반 행": violations(raw),
        "중립 F1 (규칙 전)": f1_score(labels["test"][:, NEUTRAL], raw[:, NEUTRAL], zero_division=0),
        "중립 F1 (규칙 후)": f1_score(labels["test"][:, NEUTRAL], after[:, NEUTRAL], zero_division=0),
        "판정 없음 비율": (after.sum(axis=1) == 0).mean(),
    })
pd.DataFrame(rows).round(4)

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["axes.unicode_minus"] = False
if IN_COLAB:
    !apt-get -qq install fonts-nanum > /dev/null
    import matplotlib.font_manager as fm
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    plt.rcParams["font.family"] = "NanumGothic"

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(CATEGORIES))
for i, (row, color) in enumerate(zip(summary[:2], ["#8a94a6", "#d9622b"])):
    ax.bar(x + (i - 0.5) * 0.38, [row[f"test_f1_{c}"] for c in CATEGORIES], width=0.38, label=row["설정"], color=color)
ax.set_xticks(x, CATEGORIES)
ax.set_ylabel("test F1")
ax.set_ylim(0, 1)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 3. 리뷰 예측 함수와 정성 데모 (GPU 권장)

최종 모델 폴더만 있으면 모델과 튜닝된 threshold를 함께 불러온다. 아무 범주도 켜지지 않은 **판정 없음**이면, 데모 출력에서만 확률이 가장 높은 범주를 "(약함)"으로 보여준다. 지표 계산에는 이 표시가 쓰이지 않는다.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(FINAL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(FINAL_DIR).to(device).eval()
final_th = np.array([json.load(open(f"{FINAL_DIR}/thresholds.json"))[c] for c in CATEGORIES])
assert np.allclose(final_th, THRESHOLDS)  # 저장한 threshold를 그대로 다시 읽는다

@torch.no_grad()
def predict_probs(texts, batch_size=128):
    out = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(texts[i:i + batch_size], truncation=True, max_length=MAX_LEN, padding=True, return_tensors="pt").to(device)
        out.append(torch.sigmoid(model(**enc).logits).float().cpu().numpy())
    return np.concatenate(out)

def describe(prob_row, pred_row):
    on = [f"{CATEGORIES[j]} {prob_row[j]:.2f}" for j in np.flatnonzero(pred_row)]
    if on:
        return ", ".join(on)
    j = int(prob_row.argmax())
    return f"{CATEGORIES[j]} {prob_row[j]:.2f} (약함)"   # 판정 없음: 데모 표시만, 지표에는 안 씀

def predict_reviews(texts):
    probs = predict_probs(list(texts))
    preds = predict_labels(probs, final_th)
    return probs, preds, [describe(p, y) for p, y in zip(probs, preds)]

# 다시 불러온 모델이 03이 저장한 확률과 같은 예측을 낸다 (모델·threshold 저장이 맞다)
val_texts = kote["val"].loc[final["val_ids"][:32], "text"].astype(str).tolist()
assert np.allclose(predict_probs(val_texts), final["val_probs"][:32], atol=1e-2)  # 03은 GPU fp16으로 예측했을 수 있다

In [ ]:
NSMC_URL = "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt"
nsmc = pd.read_csv(NSMC_URL, sep="\t", quoting=3).dropna(subset=["document"])
POLARITY = {0: "부정", 1: "긍정"}

hand_picked = pd.DataFrame({"document": [
    "배우들 연기가 미쳤다. 두 번 봐도 또 보고 싶음",   # 01 데모 리뷰
    "시간 아깝다 진짜 노잼",
    "결말이 너무 어이없어서 화가 난다",
    "음악이 좋아서 끝까지 봤다",
    "마지막 장면에서 펑펑 울었다",
    "중간에 깜짝 놀라서 소리 질렀음",
], "label": [1, 0, 0, 1, 1, 1]})
n_demo = 3 if SMOKE else 15
demo = pd.concat([hand_picked, nsmc.groupby("label").sample(n_demo, random_state=42)], ignore_index=True)

demo_probs, demo_preds, demo["예측 감정 범주"] = predict_reviews(demo["document"])
demo["NSMC 극성"] = demo["label"].map(POLARITY)

# "(약함)"은 판정 없음 행에만 붙고, 지표용 예측(demo_preds)은 그 행에서 여전히 비어 있다
weak = demo["예측 감정 범주"].str.endswith("(약함)").values
assert np.array_equal(weak, demo_preds.sum(axis=1) == 0)

pd.set_option("display.max_colwidth", 80)
demo[["document", "NSMC 극성", "예측 감정 범주"]].rename(columns={"document": "리뷰"})

In [ ]:
# 즉석 시연: 원하는 리뷰를 넣어 본다
text = "연출은 좋은데 너무 길어서 졸렸다"
print(f"{text}\n  → {predict_reviews([text])[2][0]}")

## 4. NSMC 극성 일관성 점검 (GPU 권장)

극성 분류 지식과 감정 분류가 맞물리는지 본다: 긍정 리뷰에는 **기쁨**이, 부정 리뷰에는 **분노·슬픔·혐오·지루함** 중 하나가 켜지기를 기대한다.

In [ ]:
POS_FAMILY = ["기쁨"]
NEG_FAMILY = ["분노", "슬픔", "혐오", "지루함"]
n_check = 20 if SMOKE else 2000
check = nsmc.groupby("label").sample(n_check, random_state=0).reset_index(drop=True)
_, check_preds, check["예측 감정 범주"] = predict_reviews(check["document"])

idx = lambda fam: [CATEGORIES.index(c) for c in fam]
check["기쁨 계열"] = check_preds[:, idx(POS_FAMILY)].any(axis=1)
check["부정 계열"] = check_preds[:, idx(NEG_FAMILY)].any(axis=1)
check["판정 없음"] = check_preds.sum(axis=1) == 0
check["NSMC 극성"] = check["label"].map(POLARITY)

consistency = check.groupby("NSMC 극성").agg(
    리뷰_수=("label", "size"),
    기쁨_계열=("기쁨 계열", "mean"),
    부정_계열=("부정 계열", "mean"),
    둘_다=("기쁨 계열", lambda s: (s & check.loc[s.index, "부정 계열"]).mean()),
    판정_없음=("판정 없음", "mean"),
)
save_result(consistency, "final_consistency.csv")
print(f"저장: {RESULTS_DIR}/final_*.csv (Colab이면 Drive {RUNS_DIR}/final_*.csv 를 내려받아 저장소 outputs/results/ 에 커밋)")
consistency.round(3)

In [ ]:
# 극성별 감정 범주 분포
dist = pd.DataFrame(check_preds, columns=CATEGORIES).groupby(check["NSMC 극성"]).mean()
dist.round(3)

In [ ]:
# 어긋난 예시: 긍정인데 부정 계열만, 부정인데 기쁨만 켜진 리뷰
mismatch = check[((check["label"] == 1) & check["부정 계열"] & ~check["기쁨 계열"]) |
                 ((check["label"] == 0) & check["기쁨 계열"] & ~check["부정 계열"])]
print(f"어긋난 리뷰 {len(mismatch)}건 / {len(check)}건")
mismatch.groupby("NSMC 극성").head(5)[["document", "NSMC 극성", "예측 감정 범주"]].rename(columns={"document": "리뷰"})

## 5. 해석 (Colab 전체 실행 후 작성)

- 최종 모델이 2단계 학습인 이유: 비교 결과와 관계없이 정해 둔 결정 (GLOSSARY)
- 03의 범주별 차이 표에서 2단계 학습이 이기고 진 범주와 그 원인
- threshold 튜닝으로 소수 범주(지루함·중립)가 얼마나 살아났는지, 그리고 그 threshold가 val 소수 표본(중립 약 150건)에서 정해져 불안정할 수 있다는 점
- 데이터 한계: KOTE 라벨 잡음, 영화 리뷰는 KOTE 12개 출처 중 하나, 2단계 학습 3시드가 같은 NSMC 모델에서 출발